# 🤖 Building Agents with Agent Framework

## 🎯 What You'll Learn

In this notebook, you'll learn how to build **AI agents** using Microsoft's Agent Framework. Agents are autonomous AI systems that can use tools, make decisions, and complete complex tasks.

### 🤖 **What are AI Agents?**
- **Autonomous**: Can work independently to achieve goals
- **Tool-Using**: Can interact with external systems and APIs
- **Decision-Making**: Can reason about what actions to take
- **Goal-Oriented**: Work towards completing specific objectives

### 🎯 **Why Agent Framework?**
- 🚀 **Built for Agents** - Designed specifically for agent workflows
- 🔧 **Rich Tooling** - Pre-built tools and easy custom tool creation
- 🔄 **Conversation Management** - Handles multi-turn interactions
- 📊 **Observability** - Built-in tracing and monitoring

### 🛠️ **What We'll Build**
Three different types of agents:
1. **🧮 Calculator Agent** - Performs mathematical calculations
2. **📝 Writing Assistant Agent** - Helps with text analysis and writing
3. **🌐 Research Agent** - Simulates web research capabilities

---

## 🚀 Let's Build Some Agents!

### 🔐 **Environment Configuration**

**Option A: Using GitHub Models (Recommended for Learning)**
```env
GITHUB_TOKEN=your_github_personal_access_token
```

**Option B: Using Azure OpenAI**
```env
AZURE_OPENAI_API_KEY=your_api_key_here
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_DEPLOYMENT_NAME=your_gpt_deployment_name
```

> **💡 Tip:** Create a `.env` file in your project root with these variables.

---

In [ ]:
# 📦 Install Required Packages
%pip install agent-framework python-dotenv --quiet

print("✅ Agent Framework installed successfully!")
print("🤖 Ready to build intelligent agents!")

In [ ]:
# 🔧 Setup and Configuration
import os
import json
import math
from typing import Annotated
from pydantic import Field
from dotenv import load_dotenv

# Import Agent Framework components
from agent_framework.openai import OpenAIChatClient
from agent_framework.azure import AzureOpenAIChatClient
from agent_framework.observability import setup_observability

# Load environment variables
load_dotenv()

# 📊 Setup observability (optional - for production monitoring)
try:
    setup_observability(
        otlp_endpoint="http://localhost:4317",
        enable_sensitive_data=True
    )
    observability_enabled = True
except:
    observability_enabled = False

# 🔗 Setup chat client
def create_chat_client():
    """Create a chat client using GitHub Models or Azure OpenAI."""
    try:
        if os.getenv("GITHUB_TOKEN"):
            return OpenAIChatClient(
                endpoint="https://models.inference.ai.azure.com",
                model="gpt-4o-mini",
                api_key=os.getenv("GITHUB_TOKEN")
            )
        elif os.getenv("AZURE_OPENAI_API_KEY"):
            return AzureOpenAIChatClient(
                endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
                deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),  # Changed from 'model' to 'deployment_name'
                api_key=os.getenv("AZURE_OPENAI_API_KEY")
            )
        else:
            raise ValueError("Please set either GITHUB_TOKEN or AZURE_OPENAI_API_KEY")
    except Exception as e:
        print(f"⚠️ Error setting up chat client: {e}")
        raise

chat_client = create_chat_client()

print("✅ Agent Framework configured")
print("💬 Chat client ready")
print("🤖 Ready to build agents!")

## 🧮 Agent 1: Calculator Agent

Let's build our first agent! This agent can perform mathematical operations by using tools.

### 🔍 What This Agent Does:
- **Understands** math questions in natural language
- **Decides** which calculation tools to use
- **Executes** the calculations using Python
- **Explains** the results in a friendly way

This is a **real agent** calling Azure OpenAI - not a mock!

## 🧠 How Agents Actually Work

Before we build our agents, let's understand what's happening behind the scenes:

### 🔄 The Agent Loop

When you ask an agent a question, here's what happens:

1. **🎯 Question Analysis** - The LLM (GPT-4) analyzes your question
2. **🛠️ Tool Selection** - The agent decides which tools to use (if any)
3. **⚡ Tool Execution** - The selected tools run with appropriate parameters
4. **🧠 Reasoning** - The LLM processes tool results and reasons about next steps
5. **💬 Response Generation** - The agent generates a final answer

### 🤖 Key Points

- **Real LLM Calls**: Every agent call hits Azure OpenAI or GitHub Models
- **Autonomous Reasoning**: The LLM decides when and how to use tools
- **Multi-Step Thinking**: Agents can use multiple tools in sequence
- **No Mocking**: All demonstrations use real AI models!

Let's see this in action! 👇

In [ ]:
# 🧮 Build Calculator Agent

# ═══════════════════════════════════════════════════════════════
# STEP 1: Define Tool Functions
# ═══════════════════════════════════════════════════════════════
# These are plain Python functions that the agent can call.
# The Annotated type hints tell the LLM what each parameter does!

def calculate(expression: Annotated[str, Field(description="Mathematical expression to evaluate (e.g., '2 + 3 * 4')")]) -> str:
    """Safely evaluate mathematical expressions.
    
    This tool lets the agent perform calculations. The LLM will:
    1. Extract the math expression from user's question
    2. Call this function with the expression
    3. Use the result to answer the user
    """
    try:
        # Safe evaluation - only allow mathematical operations
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            return "Error: Expression contains invalid characters"
        
        result = eval(expression)
        return f"The result of '{expression}' is {result}"
    except Exception as e:
        return f"Error calculating '{expression}': {str(e)}"

def advanced_math(
    operation: Annotated[str, Field(description="The operation to perform: sqrt, sin, cos, tan, log, or exp")],
    value: Annotated[float, Field(description="The input value for the operation")]
) -> str:
    """Perform advanced mathematical operations.
    
    The LLM can choose this tool for sqrt, trig, log, etc.
    It will extract the operation name and value from the question.
    """
    try:
        if operation == "sqrt":
            result = math.sqrt(value)
        elif operation == "sin":
            result = math.sin(math.radians(value))  # Convert to radians
        elif operation == "cos":
            result = math.cos(math.radians(value))
        elif operation == "tan":
            result = math.tan(math.radians(value))
        elif operation == "log":
            result = math.log(value)
        elif operation == "exp":
            result = math.exp(value)
        else:
            return f"Unknown operation: {operation}"
        
        return f"{operation}({value}) = {result}"
    except Exception as e:
        return f"Error performing {operation}({value}): {str(e)}"

# ═══════════════════════════════════════════════════════════════
# STEP 2: Create the Agent
# ═══════════════════════════════════════════════════════════════
# This connects to Azure OpenAI and gives it our tools!

calculator_agent = chat_client.create_agent(
    instructions=(
        "You are a helpful mathematical assistant that can perform calculations "
        "and advanced math operations. Use the provided tools to help users with "
        "mathematical problems. Always explain your reasoning and show your work."
    ),
    name="CalculatorAgent",
    tools=[calculate, advanced_math]  # Give the LLM access to our functions
)

print("✅ Calculator Agent created!")
print("🧮 Can perform basic and advanced mathematical operations")
print("🔧 Tools: calculate, advanced_math")
print("\n🔗 **Connection Status:**")
print(f"   • Connected to: {type(chat_client).__name__}")
print(f"   • Using real LLM: Yes (GPT-4 or compatible)")
print(f"   • Tool count: 2 functions")
print("\n💡 The agent will call Azure OpenAI for EVERY question!")

### 🏗️ Agent Architecture

Here's what we're creating:

```
User Question
     ↓
┌────────────────────────────────────┐
│   Agent Framework                  │
│  ┌──────────────────────────────┐  │
│  │  Azure OpenAI (GPT-4)        │  │ ← Real LLM doing reasoning
│  │  - Analyzes question         │  │
│  │  - Plans tool usage          │  │
│  │  - Generates response        │  │
│  └──────────────────────────────┘  │
│             ↕                      │
│  ┌──────────────────────────────┐  │
│  │  Tool Functions              │  │ ← Your Python functions
│  │  • calculate(expression)     │  │
│  │  • advanced_math(op, value)  │  │
│  └──────────────────────────────┘  │
└────────────────────────────────────┘
     ↓
Answer with tool results
```

The agent **orchestrates** between the LLM and your tools automatically!

In [ ]:
# 🧪 Test Calculator Agent
print("🧪 **Testing Calculator Agent with REAL Azure OpenAI**")
print("=" * 60)

# Test basic calculation
question1 = "What is 15 * 8 + 42?"
print(f"❓ **Question:** {question1}\n")

print("🧠 **What's happening behind the scenes:**")
print("   1. Agent sends question to Azure OpenAI (GPT-4)")
print("   2. LLM analyzes: 'This needs the calculate tool'")
print("   3. LLM extracts expression: '15 * 8 + 42'")
print("   4. Agent executes calculate('15 * 8 + 42')")
print("   5. Tool returns: 'The result of 15 * 8 + 42 is 162'")
print("   6. LLM formats friendly response\n")

response1 = await calculator_agent.run(question1)
print(f"🤖 **Agent Response:** {response1.text}\n")

# Test advanced math
question2 = "Calculate the square root of 144 and the sine of 30 degrees"
print(f"❓ **Question:** {question2}\n")

print("🧠 **Agent's reasoning:**")
print("   1. LLM identifies TWO operations needed")
print("   2. Calls advanced_math('sqrt', 144)")
print("   3. Calls advanced_math('sin', 30)")  
print("   4. Synthesizes both results into answer\n")

response2 = await calculator_agent.run(question2)
print(f"🤖 **Agent Response:** {response2.text}")
print("=" * 60)

print("\n💡 **Key Insight:** The agent is making REAL API calls to Azure OpenAI!")
print("   Each run costs tokens and uses the LLM for reasoning.")

### 🔬 Deep Dive: What Happens During an Agent Call

Let's trace through exactly what happens when you ask the agent a question:

**User asks:** "What is 15 * 8 + 42?"

1. **🌐 API Call to Azure OpenAI** - Your question is sent to GPT-4
2. **🧠 LLM Reasoning** - The model thinks:
   - "This is a math calculation"
   - "I should use the `calculate` tool"
   - "The expression is: 15 * 8 + 42"
3. **🛠️ Tool Call** - Agent Framework executes: `calculate('15 * 8 + 42')`
4. **⚡ Tool Returns** - Function returns: "The result of '15 * 8 + 42' is 162"
5. **🌐 Second API Call** - Tool result sent back to GPT-4
6. **💬 Final Response** - LLM generates friendly answer using the tool result

**Total:** 2 API calls to Azure OpenAI, ~1000 tokens used

This is **not mocked** - every step uses real infrastructure!

## 📝 Agent 2: Writing Assistant Agent

Now let's create a writing assistant agent that can help with text analysis, grammar checking, and content improvement.

In [ ]:
# 📝 Build Writing Assistant Agent

# Define tool functions
def analyze_text(text: Annotated[str, Field(description="The text to analyze")]) -> str:
    """Analyze text and provide statistics.
    
    Args:
        text: The text to analyze
        
    Returns:
        Analysis results including word count, character count, etc.
    """
    try:
        words = text.split()
        sentences = text.count('.') + text.count('!') + text.count('?')
        paragraphs = len([p for p in text.split('\n\n') if p.strip()])
        
        analysis = {
            "characters": len(text),
            "characters_no_spaces": len(text.replace(' ', '')),
            "words": len(words),
            "sentences": sentences,
            "paragraphs": paragraphs,
            "avg_words_per_sentence": round(len(words) / max(sentences, 1), 2)
        }
        
        result = "📊 **Text Analysis Results:**\n"
        result += f"• Characters: {analysis['characters']} (including spaces)\n"
        result += f"• Characters: {analysis['characters_no_spaces']} (excluding spaces)\n"
        result += f"• Words: {analysis['words']}\n"
        result += f"• Sentences: {analysis['sentences']}\n"
        result += f"• Paragraphs: {analysis['paragraphs']}\n"
        result += f"• Average words per sentence: {analysis['avg_words_per_sentence']}"
        
        return result
    except Exception as e:
        return f"Error analyzing text: {str(e)}"

def extract_keywords(
    text: Annotated[str, Field(description="The text to extract keywords from")],
    max_keywords: Annotated[int, Field(description="Maximum number of keywords to return")] = 10
) -> str:
    """Extract keywords from text.
    
    Args:
        text: The text to extract keywords from
        max_keywords: Maximum number of keywords to return
        
    Returns:
        List of extracted keywords
    """
    try:
        # Simple keyword extraction based on word frequency
        import re
        
        # Remove punctuation and convert to lowercase
        words = re.findall(r'\b\w+\b', text.lower())
        
        # Filter out common stop words
        stop_words = {'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should', 'may', 'might', 'must', 'can', 'this', 'that', 'these', 'those', 'i', 'you', 'he', 'she', 'it', 'we', 'they', 'me', 'him', 'her', 'us', 'them'}
        
        filtered_words = [word for word in words if word not in stop_words and len(word) > 2]
        
        # Count word frequency
        word_freq = {}
        for word in filtered_words:
            word_freq[word] = word_freq.get(word, 0) + 1
        
        # Sort by frequency and get top keywords
        sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
        keywords = [word for word, freq in sorted_words[:max_keywords]]
        
        return f"🔑 **Top {len(keywords)} Keywords:** {', '.join(keywords)}"
    except Exception as e:
        return f"Error extracting keywords: {str(e)}"

def check_readability(text: Annotated[str, Field(description="The text to assess for readability")]) -> str:
    """Provide basic readability assessment.
    
    Args:
        text: The text to assess
        
    Returns:
        Readability assessment and suggestions
    """
    try:
        words = text.split()
        sentences = text.count('.') + text.count('!') + text.count('?')
        
        if sentences == 0:
            return "Please provide text with complete sentences for readability analysis."
        
        avg_sentence_length = len(words) / sentences
        long_words = len([word for word in words if len(word) > 6])
        long_word_percentage = (long_words / len(words)) * 100 if words else 0
        
        assessment = "📖 **Readability Assessment:**\n"
        assessment += f"• Average sentence length: {avg_sentence_length:.1f} words\n"
        assessment += f"• Long words (6+ characters): {long_word_percentage:.1f}%\n\n"
        
        # Provide feedback
        if avg_sentence_length > 25:
            assessment += "⚠️ Consider shorter sentences for better readability\n"
        elif avg_sentence_length < 8:
            assessment += "ℹ️ Very short sentences - consider combining some\n"
        else:
            assessment += "✅ Good sentence length\n"
        
        if long_word_percentage > 20:
            assessment += "⚠️ Consider using simpler words where possible\n"
        else:
            assessment += "✅ Good vocabulary complexity\n"
        
        return assessment
    except Exception as e:
        return f"Error checking readability: {str(e)}"

# Create the Writing Assistant Agent
writing_agent = chat_client.create_agent(
    instructions="You are a helpful writing assistant that can analyze text, extract keywords, and assess readability. Use the provided tools to help users improve their writing.",
    name="WritingAssistant",
    tools=[analyze_text, extract_keywords, check_readability]
)

print("✅ Writing Assistant Agent created!")
print("📝 Can analyze text, extract keywords, and check readability")
print("🔧 Tools: analyze_text, extract_keywords, check_readability")

In [ ]:
# 🧪 Test Writing Assistant Agent
print("🧪 **Testing Writing Assistant Agent**")
print("=" * 45)

sample_text = """
Artificial intelligence is transforming the way we work and live. Machine learning algorithms can process vast amounts of data to identify patterns and make predictions. Natural language processing enables computers to understand and generate human language. These technologies are being applied in healthcare, finance, education, and many other fields to solve complex problems and improve efficiency.
""".strip()

question = f"Please analyze this text and provide insights: '{sample_text}'"
print(f"❓ **Question:** Analyze the following text about AI...")
print(f"📄 **Text:** {sample_text}")
print()

response = await writing_agent.run(question)
print(f"🤖 **Analysis:**")
print(response.text)
print("=" * 45)

## 🌐 Agent 3: Research Agent

Finally, let's create a research agent that can simulate web research capabilities and organize information.

In [ ]:
# 🌐 Build Research Agent

# Simulated knowledge base for demonstration
knowledge_base = {
    "python": {
        "definition": "Python is a high-level, interpreted programming language known for its simple syntax and versatility.",
        "uses": ["Web development", "Data science", "AI/ML", "Automation", "Scientific computing"],
        "popularity": "Very high - consistently ranked in top 3 programming languages",
        "learning_resources": ["Python.org tutorials", "Codecademy", "freeCodeCamp", "Real Python"]
    },
    "machine learning": {
        "definition": "Machine learning is a subset of AI that enables computers to learn and improve from experience without being explicitly programmed.",
        "types": ["Supervised learning", "Unsupervised learning", "Reinforcement learning"],
        "applications": ["Image recognition", "Natural language processing", "Recommendation systems", "Autonomous vehicles"],
        "popular_libraries": ["scikit-learn", "TensorFlow", "PyTorch", "Keras"]
    },
    "artificial intelligence": {
        "definition": "AI is the simulation of human intelligence in machines programmed to think and learn like humans.",
        "subfields": ["Machine Learning", "Natural Language Processing", "Computer Vision", "Robotics"],
        "current_trends": ["Large Language Models", "Generative AI", "AI Ethics", "Edge AI"],
        "impact": "Transforming industries from healthcare to finance to entertainment"
    }
}

# Define tool functions
def research_topic(topic: Annotated[str, Field(description="The topic to research")]) -> str:
    """Research a topic and provide comprehensive information.
    
    Args:
        topic: The topic to research
        
    Returns:
        Research findings about the topic
    """
    try:
        topic_lower = topic.lower()
        
        # Find matching topics in knowledge base
        matches = []
        for key, data in knowledge_base.items():
            if key in topic_lower or any(word in key for word in topic_lower.split()):
                matches.append((key, data))
        
        if not matches:
            return f"I don't have specific information about '{topic}' in my knowledge base. This would require accessing external resources."
        
        result = f"🔍 **Research Results for '{topic}':**\n\n"
        
        for key, data in matches:
            result += f"**{key.title()}:**\n"
            result += f"📖 Definition: {data.get('definition', 'Not available')}\n\n"
            
            for field, values in data.items():
                if field != 'definition':
                    if isinstance(values, list):
                        result += f"• {field.replace('_', ' ').title()}: {', '.join(values)}\n"
                    else:
                        result += f"• {field.replace('_', ' ').title()}: {values}\n"
            result += "\n"
        
        return result
    except Exception as e:
        return f"Error researching topic: {str(e)}"

def create_summary(
    information: Annotated[str, Field(description="The information to summarize")],
    max_length: Annotated[int, Field(description="Maximum length of summary in characters")] = 200
) -> str:
    """Create a summary of provided information.
    
    Args:
        information: The information to summarize
        max_length: Maximum length of summary in characters
        
    Returns:
        A concise summary
    """
    try:
        if len(information) <= max_length:
            return f"📋 **Summary:** {information}"
        
        # Simple summarization by taking first sentences up to max_length
        sentences = information.split('. ')
        summary = ""
        
        for sentence in sentences:
            if len(summary + sentence) <= max_length:
                summary += sentence + ". "
            else:
                break
        
        return f"📋 **Summary:** {summary.strip()}"
    except Exception as e:
        return f"Error creating summary: {str(e)}"

def generate_outline(topic: Annotated[str, Field(description="The topic to create an outline for")]) -> str:
    """Generate a research outline for a given topic.
    
    Args:
        topic: The topic to create an outline for
        
    Returns:
        A structured research outline
    """
    try:
        outline = f"📝 **Research Outline for '{topic}':**\n\n"
        outline += "**I. Introduction**\n"
        outline += f"   A. What is {topic}?\n"
        outline += f"   B. Why is {topic} important?\n\n"
        outline += "**II. Background and History**\n"
        outline += f"   A. Origins of {topic}\n"
        outline += f"   B. Key developments\n"
        outline += f"   C. Current state\n\n"
        outline += "**III. Key Concepts and Components**\n"
        outline += f"   A. Main principles of {topic}\n"
        outline += f"   B. Related technologies/concepts\n"
        outline += f"   C. Technical specifications\n\n"
        outline += "**IV. Applications and Use Cases**\n"
        outline += f"   A. Industry applications\n"
        outline += f"   B. Real-world examples\n"
        outline += f"   C. Benefits and advantages\n\n"
        outline += "**V. Challenges and Limitations**\n"
        outline += f"   A. Current limitations of {topic}\n"
        outline += f"   B. Common challenges\n"
        outline += f"   C. Areas for improvement\n\n"
        outline += "**VI. Future Outlook**\n"
        outline += f"   A. Emerging trends in {topic}\n"
        outline += f"   B. Future possibilities\n"
        outline += f"   C. Predictions and forecasts\n\n"
        outline += "**VII. Conclusion**\n"
        outline += f"   A. Summary of key points\n"
        outline += f"   B. Implications for the future\n"
        
        return outline
    except Exception as e:
        return f"Error generating outline: {str(e)}"

# Create the Research Agent
research_agent = chat_client.create_agent(
    instructions="You are a research assistant that can investigate topics, create summaries, and generate outlines. Use the provided tools to help users with research tasks.",
    name="ResearchAgent",
    tools=[research_topic, create_summary, generate_outline]
)

print("✅ Research Agent created!")
print("🌐 Can research topics, create summaries, and generate outlines")
print("🔧 Tools: research_topic, create_summary, generate_outline")

In [ ]:
# 🧪 Test Research Agent
print("🧪 **Testing Research Agent**")
print("=" * 40)

# Test research capability
question1 = "Research machine learning and provide an overview"
print(f"❓ **Question:** {question1}")
print()

response1 = await research_agent.run(question1)
print(f"🤖 **Research Results:**")
print(response1.text)
print("\n" + "=" * 40)

# Test outline generation
question2 = "Create a research outline for artificial intelligence"
print(f"❓ **Question:** {question2}")
print()

response2 = await research_agent.run(question2)
print(f"🤖 **Outline:**")
print(response2.text)
print("=" * 40)

## 🔄 Agent Collaboration Demo

Now let's see something really powerful: **multiple agents working together**!

### 🎯 What's Special Here:
- **Independent Agents** - Each agent is a separate AI system
- **Real Orchestration** - We manually coordinate between them
- **Complementary Skills** - Each agent has different capabilities
- **Actual LLM Calls** - Every agent call hits Azure OpenAI

### 🔬 Behind the Scenes:
This demo makes **3+ separate API calls** to Azure OpenAI:
1. Research Agent → GPT-4 analyzes + uses research tools
2. Calculator Agent → GPT-4 analyzes + uses calc tools  
3. Writing Agent → GPT-4 analyzes + uses writing tools

Each agent has its **own reasoning loop** and tool set!

In [ ]:
# 🔄 Multi-Agent Collaboration Demo
print("🔄 **Multi-Agent Collaboration Demo**")
print("=" * 60)

# Scenario: A student needs help with a data science project
print("📚 **Scenario:** A student needs help with a data science project")
print("🎯 **Goal:** Provide comprehensive assistance using multiple agents")
print("⚡ **Note:** Each agent makes REAL calls to Azure OpenAI!\n")

# Step 1: Research Agent provides background
print("═" * 60)
print("🌐 **Step 1: Research Agent - Background Information**")
print("═" * 60)
research_query = "Research Python for data science"
print(f"📝 Query: {research_query}")
print("\n🧠 What's happening:")
print("   1. Question sent to Azure OpenAI")
print("   2. LLM decides to use 'research_topic' tool")
print("   3. Tool searches knowledge base")
print("   4. LLM synthesizes results into answer\n")

research_result = await research_agent.run(research_query)
print(f"📖 Research findings (truncated):")
print(research_result.text[:300] + "...\n")

# Step 2: Calculator Agent helps with statistical concepts
print("═" * 60)
print("🧮 **Step 2: Calculator Agent - Statistical Calculations**")
print("═" * 60)
calc_query = "What is (23 + 45 + 67 + 34 + 56 + 78 + 23 + 45) / 8?"
print(f"📝 Query: {calc_query}")
print("\n🧠 What's happening:")
print("   1. NEW session with Azure OpenAI")
print("   2. Calculator agent analyzes the math problem")
print("   3. Uses 'calculate' tool to compute mean")
print("   4. Returns formatted result\n")

calc_result = await calculator_agent.run(calc_query)
print(f"🔢 Calculation result:")
print(calc_result.text + "\n")

# Step 3: Writing Agent helps with project documentation
print("═" * 60)
print("📝 **Step 3: Writing Agent - Documentation Analysis**")
print("═" * 60)
writing_query = "Analyze this project description: 'Data analysis of customer behavior patterns using Python pandas and matplotlib for visualization.'"
print(f"📝 Query: {writing_query}")
print("\n🧠 What's happening:")
print("   1. ANOTHER independent call to Azure OpenAI")
print("   2. Writing agent uses text analysis tools")
print("   3. Extracts keywords and checks readability")
print("   4. Provides comprehensive feedback\n")

writing_result = await writing_agent.run(writing_query)
print(f"📊 Writing analysis:")
print(writing_result.text + "\n")

print("═" * 60)
print("✅ **Collaboration Complete!**")
print("═" * 60)
print("\n📊 **Statistics:**")
print(f"   • Total agents used: 3")
print(f"   • Separate LLM calls: 3+")
print(f"   • Tools invoked: 5+")
print(f"   • Total tokens used: ~3000-5000")
print("\n🎯 Each agent contributed their specialized expertise")
print("🔗 Together they provided comprehensive project assistance")
print("\n💡 **This is real AI orchestration - not a simulation!**")

## 🎉 Congratulations! You Built Multiple AI Agents!

### 🏆 What You Accomplished

You've successfully created **three specialized AI agents** using the Agent Framework:

**🧮 Calculator Agent:**
- Performs mathematical calculations and advanced operations
- Demonstrates tool creation and safe code execution
- Shows how agents can handle specific domain tasks

**📝 Writing Assistant Agent:**
- Analyzes text and provides detailed statistics
- Extracts keywords and assesses readability
- Demonstrates natural language processing capabilities

**🌐 Research Agent:**
- Researches topics and provides comprehensive information
- Creates summaries and generates structured outlines
- Shows how agents can organize and present information

### 🔧 Key Agent Framework Concepts Learned

**🛠️ Tool Creation:**
- How to create custom tools for agents
- Proper function signatures and documentation
- Error handling and validation

**🤖 Agent Configuration:**
- Setting up agents with specific roles and capabilities
- Integrating tools with agent workflows
- Managing agent conversations and responses

**🔄 Multi-Agent Collaboration:**
- How different agents can work together
- Complementary skills and capabilities
- Workflow orchestration possibilities

### 🚀 Next Steps

**Enhance Your Agents:**
- 🔗 **Add External APIs** - Connect to real web services
- 💾 **Persistent Memory** - Store conversation history
- 🎯 **Specialized Tools** - Create domain-specific capabilities
- 🔄 **Advanced Workflows** - Build complex multi-step processes

**Production Considerations:**
- 🔒 **Security** - Input validation and safe execution
- 📊 **Monitoring** - Use built-in telemetry for production insights
- ⚡ **Performance** - Optimize for speed and resource usage
- 🔄 **Reliability** - Error handling and graceful degradation

### 💡 Real-World Applications

Your agent-building skills can be applied to:
- **📊 Data Analysis Assistants** - Automated reporting and insights
- **🎓 Educational Tutors** - Personalized learning experiences
- **💼 Business Process Automation** - Streamlined workflows
- **🔧 Technical Support** - Intelligent troubleshooting
- **📝 Content Creation** - Writing and editing assistance

---